# Fuel Gauging Analysis Pipeline

This notebook walks through the full analysis workflow for fuel quantity
indication systems. It works with both synthetic (simulated) data and
real aircraft H-V table data.

**Sections:**
1. Setup & Import
2. Data Exploration
3. H-V Table Validation
4. Interpolation Verification
5. Test Data Comparison
6. Error Decomposition
7. Dashboard Generation
8. Parameter Study

## 1. Setup & Import

Import the analysis toolkit and load data. This example uses the synthetic
bridge, but you can swap in `load_mat_tables()` or `load_test_csv()` for
real aircraft data.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Add project root to path so imports work from the notebooks/ directory
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Analysis toolkit imports
from synthetic_fuel_system.analysis.importers.base import HVTable, TankData, FuelSystemData
from synthetic_fuel_system.analysis.importers.synthetic_bridge import load_synthetic_system
from synthetic_fuel_system.analysis.interpolation import HVInterpolator
from synthetic_fuel_system.analysis.comparison import compute_residuals, compute_error_statistics
from synthetic_fuel_system.analysis.error_analysis import decompose_errors, attitude_sensitivity, fuel_level_sensitivity
from synthetic_fuel_system.analysis.dashboard import generate_dashboard

# For real data, use these instead:
# from synthetic_fuel_system.analysis.importers.mat_importer import load_mat_tables
# from synthetic_fuel_system.analysis.importers.csv_importer import load_test_csv

# Load synthetic system (tables + defuel/refuel sequences)
data = load_synthetic_system()
print(f"Loaded {data.n_tanks} tanks: {data.tank_ids}")
print(f"Test sequences: {len(data.test_data)}")
for ds in data.test_data:
    print(f"  {ds.sequence_type}: {ds.n_records} records, {len(ds.scale_checkpoints)} scale checkpoints")

## 2. Data Exploration

Examine the H-V table structure: how many points per table, height ranges,
volume ranges, and which attitudes are available.

In [ ]:
# Summary of each tank's H-V data
for tid in data.tank_ids:
    tank = data.tanks[tid]
    level_table = tank.get_table(0.0, 0.0)

    print(f"\n{tid} ({tank.name}) — {tank.probe_type}")
    print(f"  Attitudes: {tank.n_attitudes}")
    print(f"  Pitch range: {tank.pitch_values[0]:.0f} to {tank.pitch_values[-1]:.0f} deg")
    print(f"  Roll range:  {tank.roll_values[0]:.0f} to {tank.roll_values[-1]:.0f} deg")
    print(f"  Max volume:  {tank.max_volume:.0f} in³")
    if level_table:
        print(f"  Level table:  {level_table.n_points} points, "
              f"height {level_table.height_range[0]:.1f}..{level_table.height_range[1]:.1f} in")

## 3. H-V Table Validation

Plot H-V curves at level attitude for all tanks, and overlay curves at
different pitch/roll to visualize attitude sensitivity.

In [ ]:
# H-V curves at level attitude for all tanks
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, tid in enumerate(data.tank_ids):
    ax = axes[i]
    tank = data.tanks[tid]

    # Plot level (0,0) curve
    t0 = tank.get_table(0.0, 0.0)
    if t0:
        ax.plot(t0.heights, t0.volumes, 'b-', linewidth=2, label='Level')

    # Overlay a few tilted attitudes
    for pitch, roll, color, ls in [(3.0, 0.0, 'r', '--'), (-3.0, 0.0, 'g', '--'),
                                     (0.0, 5.0, 'orange', ':'), (0.0, -5.0, 'purple', ':')]:
        t = tank.get_table(pitch, roll)
        if t:
            ax.plot(t.heights, t.volumes, color=color, linestyle=ls, linewidth=1,
                    label=f'P={pitch:.0f} R={roll:.0f}')

    ax.set_title(f'{tid} ({tank.name})')
    ax.set_xlabel('Height (in)')
    ax.set_ylabel('Volume (in³)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# Hide unused subplot
if len(data.tank_ids) < len(axes):
    for j in range(len(data.tank_ids), len(axes)):
        axes[j].set_visible(False)

plt.suptitle('H-V Curves: Level vs Tilted Attitudes', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Interpolation Verification

Verify that the HVInterpolator produces smooth results between grid
points. Compare interpolated values at grid points against the raw
table values (should match exactly).

In [ ]:
# Interpolation verification: compare at grid points vs between grid points
tid = 'T3'  # Center tank — most complex (blend zone)
tank = data.tanks[tid]
interp = HVInterpolator(tank)

# Test at exact grid points
t0 = tank.get_table(0.0, 0.0)
max_err = 0.0
for h, v_expected in zip(t0.heights, t0.volumes):
    v_interp = interp.lookup(h, 0.0, 0.0)
    err = abs(v_interp - v_expected)
    max_err = max(max_err, err)

print(f"Max interpolation error at grid points: {max_err:.6f} in³ (should be ~0)")

# Sweep height at several attitudes (smooth curve check)
fig, ax = plt.subplots(figsize=(10, 5))
h_sweep = np.linspace(t0.heights[0], t0.heights[-1], 200)

for pitch, roll, label in [(0, 0, 'Level'), (1.5, -2.5, 'P=1.5/R=-2.5'),
                             (-3, 4, 'P=-3/R=4')]:
    volumes = [interp.lookup(h, pitch, roll) for h in h_sweep]
    ax.plot(h_sweep, volumes, label=label, linewidth=1.5)

    # Overlay exact grid-point values if available
    t = tank.get_table(pitch, roll)
    if t:
        ax.scatter(t.heights, t.volumes, s=15, zorder=5)

ax.set_xlabel('Height (in)')
ax.set_ylabel('Volume (in³)')
ax.set_title(f'{tid} — Interpolation Smoothness Check')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Test Data Comparison

Compare indicated fuel weight against scale reference weight. Compute
residuals and plot the timeseries.

In [ ]:
# Compute residuals for the defuel sequence
ds_defuel = data.test_data[0]
residuals = compute_residuals(ds_defuel)
stats = compute_error_statistics(residuals)

print("Error Statistics (Defuel):")
print(f"  Mean:  {stats['mean_error_lb']:+.2f} lb")
print(f"  RMS:   {stats['rms_error_lb']:.2f} lb")
print(f"  Max:   {stats['max_error_lb']:.2f} lb")
print(f"  P50:   {stats['percentiles'][50]:.2f} lb")
print(f"  P95:   {stats['percentiles'][95]:.2f} lb")
print(f"  P99:   {stats['percentiles'][99]:.2f} lb")

# Plot indicated vs actual with residual subplot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), height_ratios=[3, 1], sharex=True)

ax1.plot(residuals['time_s'], residuals['indicated_total_lb'], label='Indicated', linewidth=1)
ax1.plot(residuals['time_s'], residuals['actual_total_lb'], label='Actual (scale)', linewidth=1)
ax1.set_ylabel('Fuel Weight (lb)')
ax1.legend()
ax1.set_title('Defuel Sequence — Indicated vs Actual Weight')
ax1.grid(True, alpha=0.3)

ax2.plot(residuals['time_s'], residuals['residual_total_lb'], 'r-', linewidth=0.8)
ax2.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Error (lb)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Error Decomposition

Break down the total error into density vs volume contributions, and
examine sensitivity to attitude and fuel level.

In [ ]:
# Error decomposition: density vs volume
decomp = decompose_errors(residuals, density_lab=6.71)
print(f"Error Attribution:")
print(f"  Density contribution: {decomp['density_fraction']:.1%}")
print(f"  Volume contribution:  {decomp['volume_fraction']:.1%}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Decomposition timeseries
ax = axes[0]
ax.plot(residuals['time_s'], decomp['density_error_lb'], label='Density error', alpha=0.8)
ax.plot(residuals['time_s'], decomp['volume_error_lb'], label='Volume error', alpha=0.8)
ax.plot(residuals['time_s'], decomp['total_error_lb'], 'k--', label='Total', linewidth=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Error (lb)')
ax.set_title('Error Decomposition')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Fuel level sensitivity
level_sens = fuel_level_sensitivity(residuals, n_bins=15)
ax = axes[1]
ax.bar(level_sens['bin_centers'], level_sens['mean_abs_error'],
       width=np.diff(level_sens['bin_edges']).mean() * 0.8, alpha=0.7, color='steelblue')
ax.errorbar(level_sens['bin_centers'], level_sens['mean_abs_error'],
            yerr=level_sens['std_error'], fmt='none', color='navy', capsize=3)
ax.set_xlabel('Actual Fuel Weight (lb)')
ax.set_ylabel('Mean |Error| (lb)')
ax.set_title('Error vs Fuel Level')
ax.grid(True, alpha=0.3)

# Attitude sensitivity heatmap
att_sens = attitude_sensitivity(residuals, pitch_bins=10, roll_bins=10)
ax = axes[2]
im = ax.pcolormesh(att_sens['roll_edges'], att_sens['pitch_edges'],
                   att_sens['mean_abs_error'], cmap='YlOrRd', shading='flat')
plt.colorbar(im, ax=ax, label='Mean |Error| (lb)')
ax.set_xlabel('Roll (deg)')
ax.set_ylabel('Pitch (deg)')
ax.set_title('Attitude Sensitivity')

plt.tight_layout()
plt.show()

## 7. Dashboard Generation

Generate a standalone HTML dashboard with all 4 interactive views.
Open the resulting file in a browser.

In [ ]:
# Generate the interactive HTML dashboard
output_path = str(Path.cwd().parent / "plots" / "dashboard.html")
dashboard_path = generate_dashboard(
    data, residuals, output_path,
    title="Synthetic Fuel System — Gauging Analysis",
    max_attitudes=50,
)

import os
size_kb = os.path.getsize(dashboard_path) / 1024
print(f"Dashboard saved: {dashboard_path}")
print(f"File size: {size_kb:.0f} KB")
print("Open this file in a web browser to explore the interactive views.")

## 8. Parameter Study

Compare defuel vs refuel error characteristics side-by-side.
Useful for identifying asymmetric error sources (e.g., hysteresis,
direction-dependent blend zone behavior).

In [ ]:
# Compare defuel vs refuel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, ds in enumerate(data.test_data):
    res = compute_residuals(ds)
    st = compute_error_statistics(res)
    ax = axes[i]

    ax.plot(res['time_s'], res['residual_total_lb'], linewidth=0.8)
    ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Error (lb)')
    ax.set_title(f'{ds.sequence_type.title()} — '
                 f'RMS={st["rms_error_lb"]:.1f} lb, Max={st["max_error_lb"]:.1f} lb')
    ax.grid(True, alpha=0.3)

    # Print comparison
    print(f"\n{ds.sequence_type.upper()}:")
    print(f"  Mean: {st['mean_error_lb']:+.2f} lb | RMS: {st['rms_error_lb']:.2f} lb | "
          f"Max: {st['max_error_lb']:.2f} lb | P95: {st['percentiles'][95]:.2f} lb")

plt.suptitle('Defuel vs Refuel Error Comparison', fontsize=14)
plt.tight_layout()
plt.show()